# Get Syllables

In [1]:
import helper.library as library
import helper.syllab as syllab

In [2]:
import time
import pandas as pd

In [3]:
# syllab.py
from enum import Enum
from helper import library
from string import punctuation

# define the possible types of letters
class Letters(Enum):
    P = -2  # punctuation
    SP = -1 # space
    UNK = 0 # unknown char
    C = 1   # consonant
    M = 2   # m (could be syllabic or consonant)
    N = 3   # n (could be syllabic, consonant, or nasal vowel)
    V = 4   # vowel

# helper functions
def ischar(text):
    return text[0].isalpha()

def chartype(text):
    if ischar(text): 
        if text[0] in ['a', 'e', 'i', 'o', 'u']: return Letters.V
        elif text[0] == 'm': return Letters.M
        elif text[0] == 'n': return Letters.N
        else: return Letters.C
    elif text[0] in set(punctuation): return Letters.P
    elif text[0].isspace(): return Letters.SP
    else: return Letters.UNK

# syllabify a list of letters, assuming len(letters) != 0
def get_next_syll(letters, syllables, print_it=False):    
    # get types of letters
    # get last char's type
    types = [chartype(letters[-1])]
    
    # get second and third to last chars (if present)
    if len(letters) > 2:
        types.insert(0, chartype(letters[-2]))
        types.insert(0, chartype(letters[-3]))
    # get second to last char, default third to last to SP
    elif len(letters) > 1: 
        types.insert(0, chartype(letters[-2]))
        types.insert(0, Letters.SP)
    # set other values to default value SP
    else:
        types.insert(0, Letters.SP)
        types.insert(0, Letters.SP)
    
    if print_it: print('Types:', types)

    # identify the syllable
    # logic based on Kumolalo 2010
    curr_syll = [] # store current identified syllable
    # handle when last thing isn't a letter
    if (types[-1] == Letters.SP) or (types[-1] == Letters.P):
        curr_syll = ['SP', letters[-1]]
        letters = letters[:-1]
        if print_it: print('SP curr_syll : ', curr_syll)
    elif (types[-1] == Letters.UNK):
        curr_syll = ['UNK', letters[-1]]
        letters = letters[:-1]
        if print_it: print('UNK curr_syll : ', curr_syll)

    # look for CVn (same as DVn in this set up)
    elif (types[-3] in [Letters.C, Letters.M, Letters.N]) and (types[-2] == Letters.V) and (types[-1] == Letters.N):
        curr_syll = letters[-3:]
        letters = letters[:-3]
        if print_it: print('CVn curr_syll : ', curr_syll)
    # look for CV (same as DV)
    elif (types[-2] in [Letters.C, Letters.M, Letters.N]) and (types[-1] == Letters.V):
        curr_syll = letters[-2:]
        letters = letters[:-2]
        if print_it: print('CV curr_syll : ', curr_syll)
    # look for Vn
    elif (types[-2] == Letters.V) and (types[-1] == Letters.N):
        curr_syll = letters[-2:]
        letters = letters[:-2]
        if print_it: print('Vn curr_syll : ', curr_syll)
    # look for V
    elif types[-1] == Letters.V:
        curr_syll = letters[-1:]
        letters = letters[:-1]
        if print_it: print('V curr_syll : ', curr_syll)
    # look for N
    elif types[-1] in [Letters.N, Letters.M]:
        curr_syll = letters[-1:]
        letters = letters[:-1]
        if print_it: print('N curr_syll : ', curr_syll)
    # handle other scenario (must be an unknown syllable type)
    else:
        curr_syll = ['UNK', letters[-1]]
        letters = letters[:-1]
        if print_it: print('UNK : ', letters[-3:])

    syllables.insert(0,curr_syll) # add syllable to front of list
    return letters, syllables

def syllabify_letters(letters, print_it = False):
    syllables = []
    # get each syllable
    while (len(letters) > 0):
        if print_it: print('get_next_syll')
        letters, syllables = get_next_syll(letters, syllables)

    # merge non-standard syllables
    # prev starts out empty, then append label and all letters
    # when it ends, add non-empty one to list of syllables, and reset to empty
    merged_syllables = []
    prev_sp = []
    prev_p = []
    prev_err = []
    prev_unk = []

    for syllable in syllables:
        def _reset(prev_sp, prev_p, prev_err, prev_unk):
            # find non-empty merged non-standard syllable
            if len(prev_sp) > 0: 
                merged_syllables.append(prev_sp)
                prev_sp = []
            if len(prev_p) > 0:
                merged_syllables.append(prev_p)
                prev_p = []
            if len(prev_err) > 0:
                merged_syllables.append(prev_err)
                prev_err = []
            if len(prev_unk) > 0:
                merged_syllables.append(prev_unk)
                prev_unk = []
            return prev_sp, prev_p, prev_err, prev_unk

        # collect non-standard syllables but don't add until merged
        if syllable[0] == 'SP':
            if len(prev_sp) == 0: 
                prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk) # reset possible previous merger (only necessary for first item in SP)
                prev_sp.append('SP') # add label
            prev_sp.append(syllable[1]) # add subsequent items
        elif syllable[0] == 'P':
            if len(prev_p) == 0: 
                prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk)
                prev_p.append('P') # add label
            prev_p.append(syllable[1]) # add subsequent items
        elif syllable[0] == 'ERR':
            if len(prev_err) == 0: 
                prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk)
                prev_err.append('ERR') # add label
            prev_err.append(syllable[1]) # add subsequent items
        elif syllable[0] == 'UNK':
            if len(prev_unk) == 0: 
                prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk)
                prev_unk.append('UNK') # add label
            prev_unk.append(syllable[1]) # add subsequent items

        # add merged non-standard syllables, current syllable, and reset
        else:
            prev_sp, prev_p, prev_err, prev_unk = _reset(prev_sp, prev_p, prev_err, prev_unk) # add any merged non-standard syllables
            # add current syllable
            merged_syllables.append(syllable)
            
    return merged_syllables

def syllabify_df(df):
    syllables = []
    for id, row in df.iterrows():
        letters = library.get_letters(row['sentence'])
        curr_sylls = syllabify_letters(letters)
        syllables.append([id, curr_sylls])
    return syllables

In [4]:
iroyin_full = library.load_dataset('/Users/graven2/Documents/THESIS/data/iroyinspeech_full.csv')
multidiac_full = library.load_dataset('/Users/graven2/Documents/THESIS/data/multidiac_full.csv')
yad_full = library.load_dataset('/Users/graven2/Documents/THESIS/data/yad_full_CLEAN.csv')

In [5]:
print('---Iroyin---')
iroyin_syllables = syllabify_df(iroyin_full)
print('---MultiDiac---')
multidiac_syllables = syllabify_df(multidiac_full)
print('---YAD---')
yad_syllables = syllabify_df(yad_full)


---Iroyin---
---MultiDiac---
---YAD---
nan


In [6]:
all_syllables = pd.DataFrame(columns=['Source', 'ID', 'Sentence', 'Syllables'])
print('---IROYIN---')
for row in iroyin_syllables:
    source = 'IroyinSpeech'
    id = row[0]
    sentence = iroyin_full.loc[id, 'sentence']
    syllables = row[1]
    all_syllables.loc[len(all_syllables)] = [source, id, sentence, syllables]

print('---MultiDiac---')
for row in multidiac_syllables:
    source = 'MultiDiac'
    id = row[0]
    sentence = multidiac_full.loc[id, 'sentence']
    syllables = row[1]
    all_syllables.loc[len(all_syllables)] = [source, id, sentence, syllables]

print('---YAD---')
for row in yad_syllables:
    source = 'YAD'
    id = row[0]
    sentence = yad_full.loc[id, 'sentence']
    syllables = row[1]
    all_syllables.loc[len(all_syllables)] = [source, id, sentence, syllables]



---IROYIN---
---MultiDiac---
---YAD---


In [7]:
print(all_syllables)
all_syllables.to_csv('syllabified.csv')

             Source             ID  \
0      IroyinSpeech  yo_f_0001.wav   
1      IroyinSpeech  yo_f_0002.wav   
2      IroyinSpeech  yo_f_0003.wav   
3      IroyinSpeech  yo_f_0004.wav   
4      IroyinSpeech  yo_f_0005.wav   
...             ...            ...   
25801           YAD          20117   
25802           YAD          20118   
25803           YAD          20119   
25804           YAD          20121   
25805           YAD          20122   

                                                Sentence  \
0      Ara à mií dá ṣáká bíi ara ọmọ tuntun ní àárọ̀...   
1      Ìgbákejì Ààrẹ di adelé nígbà tí ààrẹ ti ...   
2            Ààrẹ orílẹ̀-èdè Nàìjíríà wà ní òkè òkun.   
3      Eléyìí ni ó jẹ́ ìkẹrìnléláàdọ́rín ìpàdé àp...   
4      Ọjọ́ ọ Àìkú ni aṣojú orílẹ̀-èdè Nàìjíríà fún i...   
...                                                  ...   
25801                                  Màdáámú, ẹ wo bí.   
25802  “Omijé náà ti ṣé ìran Làbákẹ́ bàìbàì, láàrin ì...   
258

# Run N-Grams

In [7]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

In [8]:
# ngrams.py
from helper import library
from enum import Enum
from helper import syllab
import pandas as pd

# Create consistent labels for tones
class Tones(Enum):
    H = 1
    M = 0
    L = -1

# possible vowels in Yoruba
VOWELS = ['a','e','i','o','u']

# remove the diacritics from a syllable
# can keep 'underdiacs', 'tone', or 'none'
def _rm_diacritics_syll(syllable, keep='underdiacs'):
    new_syll = []
    for letter in syllable:
        # corner cases
        if (letter == 'SP') or (letter == 'P'): return syllable
        if (letter == 'ERR') or (letter == 'UNK'): 
            new_syll.append(letter)
            continue
        if (letter[0:2] == 'gb'):
            new_syll.append(letter[0:2])
            continue

        # normal syllable
        include = [letter[0]] # keep original letter
                
        # check second char
        if (len(letter) > 1) and ((letter[1] in library.UNDERDIACS) and (keep=='underdiacs')):
            include.append(letter[1])
        if (len(letter) > 1) and ((letter[1] in library.TONECHARS) and (keep=='tone')):
            include.append(letter[1])

        # check third char
        if (len(letter) > 2) and ((letter[2] in library.UNDERDIACS) and (keep=='underdiacs')):
            include.append(letter[2])
        if (len(letter) > 2) and ((letter[2] in library.TONECHARS) and (keep=='tone')):
            include.append(letter[2])

        # create string from included characters
        new_syll.append(''.join(include))
    return new_syll

# remove diacritics from a set of syllables
def rm_diacritics_row(row, keep='underdiacs'):
    return [_rm_diacritics_syll(x, keep) for x in row['Syllables']]

# remove diacritics from df
# can keep 'underdiacs', 'tone', or 'none'
def rm_diacritics_df(df, keep='underdiacs'):
    new_df = df.copy()
    new_df['Syllables'] = new_df.apply(lambda row: rm_diacritics_row(row, keep), axis=1)
    return new_df

# return the index where the tone carrier is located in syllable
def _tone_carrier_index(syllable):
    # find tone carrier in syllable (either N or M alone, or V)
    if len(syllable) == 1: return 0
    
    # if len > 0, tone carrier MUST be a vowel; at most 1 vowel per syll
    else: 
        if (syllable[0][0] in VOWELS): return 0
        else: return 1 # vowel MUST be first or second in syllable

# identify the tone of a syllable
# returns 
def get_tone(syllable):
    # corner cases
    if syllable[0] == 'SP' or syllable[0] == 'P' or syllable[0] == 'ERR' or syllable[0] == 'UNK': return Tones.M

    tone_carrier = syllable[_tone_carrier_index(syllable)]

    # get tone from tone carrier
    if (len(tone_carrier) > 1) and (tone_carrier[1] in library.TONECHARS):
        tone = tone_carrier[1]
    elif (len(tone_carrier) > 2) and (tone_carrier[2] in library.TONECHARS):
        tone = tone_carrier[2]
    else: return Tones.M # no tone present --> Mid tone

    if tone == library.LOTONE: return Tones.L
    elif tone == library.HITONE: return Tones.H
    else: return Tones.M # default/error is midtone

# add a given tone to a syllable
def _add_tone(syllable, tone):
    if tone == Tones.H: tone_char = library.HITONE
    elif tone == Tones.L: tone_char = library.LOTONE
    else: return syllable # mid tone is unmarked

    index = _tone_carrier_index(syllable)
    new_syll = syllable[:]
    new_syll[index] = ''.join([syllable[index], tone_char])
    return new_syll

# find the n-sized context string for syllable i in syllables
def _get_context(syllables, n, i, keep='underdiacs'):
    context = []
    for j in range(1, n+1):
        # get -Syl
        # insert at FRONT of list
        if (i-j >= 0): 
            curr_context = syllables[i-j]
            curr_context = _rm_diacritics_syll(curr_context, keep=keep)
            if (curr_context[0] == 'SP') or (curr_context[0] == 'ERR') or (curr_context[0] == 'UNK'):
                curr_context_str = curr_context[0]
            else: curr_context_str = ''.join(curr_context)
            context.insert(0, curr_context_str)
        else: context.insert(0, '<') # start of sentence token

        # get +Syl
        # add to END of list
        if (i+j < len(syllables)):
            curr_context = syllables[i+j]
            curr_context = _rm_diacritics_syll(curr_context, keep=keep)
            if (curr_context[0] == 'SP') or (curr_context[0] == 'ERR') or (curr_context[0] == 'UNK'):
                curr_context_str = curr_context[0]
            else: curr_context_str = ''.join(curr_context)
            context.append(curr_context_str)
        else: context.append('>') # end of sentence token
    
    # merge context into a string
    context_str = '.'.join(context) # -Syl.-Syl.+Syl.+Syl
    return context_str


# get n-grams
# n is the number of items before and after to consider
def _syll_grams(syllables, counts, n, keep='underdiacs'):
    if not counts: 
        counts = []
        for i in range(n+1): counts.append(dict())
    # counts has format [n=0, n=1, ..., {syll: {-Syl.-Syl.+Syl.+Syl : {H : count, M : count, L : count}}}]
    for i, syll in enumerate(syllables):
        syll_str = ''.join(_rm_diacritics_syll(syll, keep=keep))
        # corner case
        if (syll[0] == 'SP') or (syll[0] == 'ERR') or (syll[0] == 'UNK'):
            syll_str = syll[0]

        # find all contexts in range 0-n (inclusive)
        for j in range(n+1):
            # get contexts for this syllable so far
            poss_contexts = counts[j].get(syll_str, dict())

            # get current context for syllable
            context_str = _get_context(syllables, j, i, keep=keep)

            # update with new tones
            context_tones = poss_contexts.get(context_str, dict())
            curr_tone = get_tone(syll)
            curr_tone_count = context_tones.get(curr_tone, 0)

            # update all dictionaries
            context_tones.update({curr_tone : curr_tone_count + 1})
            poss_contexts.update({context_str : context_tones})
            counts[j].update({syll_str : poss_contexts})

    return counts

# create a full n-gram count from a df
def create_syll_grams(df, n=4, keep='underdiacs'):
    counts = dict()
    for _, row in df.iterrows():
        counts = _syll_grams(row['Syllables'], counts, n, keep=keep)
    return counts

# predict the tone for each syllable in list of syllables
def pred_tone(syllables, counts, n=4, print_it=False):
    with_tones = []
    
    for i, syll in enumerate(syllables):
        # corner case
        if syll[0] == 'SP' or syll[0] == 'ERR':
            with_tones.append(syll)
            continue

        # get current syllable and its context WITH BACKOFF
        syll_str = ''.join(syll)
        new_syll = ''
        # try n, n-1, ..., 0 (and stop as soon as it is possible)
        for j in range(n, -1, -1):
            # get context
            context_str = _get_context(syllables, j, i)

            # collect stored counts
            poss_tones = counts[j].get(syll_str, dict()).get(context_str, dict())

            # get most frequent tone
            h = poss_tones.get(Tones.H, 0)
            m = poss_tones.get(Tones.M, 0)
            l = poss_tones.get(Tones.L, 0)

            # add the tone
            if (h > m) and (h > l): new_syll = _add_tone(syll, Tones.H)
            elif (l > m) and (l > h): new_syll = _add_tone(syll, Tones.L)
            else: new_syll = _add_tone(syll, Tones.M)

            # break if this syllable/sequence of syllables exists in these n-grams
            if not poss_tones: continue # dictionary is empty, so keep trying
            else: 
                if(print_it) : print('n-gram found, stopping; j = ',j,  syll_str, context_str)
                break # dictionary was found, so stop going to smaller syllables
        with_tones.append(new_syll)

    return with_tones

# do full predictions
def predict_all_tones(df, counts, n=4, print_it=False):
    new_df = df.copy()
    new_df['Prediction'] = new_df.apply(lambda row: pred_tone(row['Syllables'], counts, n, print_it), axis=1)
    return new_df

# calculate word error rate for a row, returns (wrong words, total words)
def _eval_row(row, print_it=False):
    correct = row['Syllables']
    pred = row['Prediction']

    wrong_words = 0
    total_words = 0
    in_word = False # identifies whether currently in a word or not
    curr_word_accurate = True # identifies whether the current word has gotten a tone wrong yet

    # iterate through syllables
    for i in range(len(correct)):
        # check if tones match
        corr_tone = get_tone(correct[i])
        pred_tone = get_tone(pred[i])
        if corr_tone != pred_tone: 
            curr_word_accurate = False

        # check if a word is finished
        if in_word:
            # word has ended
            if correct[i][0] == 'SP':
                in_word = False
                if not curr_word_accurate: 
                    wrong_words += 1
                    if print_it: print(f'WRONG \t{correct}\t{pred}')
                total_words += 1
                curr_word_accurate = True # reset accuracy
        if correct[i][0] != 'SP': in_word = True
    # corner case
    if in_word:
        total_words+=1
        if not curr_word_accurate: wrong_words+=1

    return pd.Series({'Wrong Words' : wrong_words, 'Total Words' : total_words})

# determine wrong words in df of syllables
def evaluate(df, print_it=False):
    new_df = df.copy()
    new_df[['Wrong Words', 'Total Words']] = new_df.apply(lambda row: _eval_row(row, print_it=print_it), axis=1, result_type='expand')
    return new_df

#


In [9]:
tester = all_syllables.loc[0, 'Syllables']
print(tester)

# for all_syllables[0]:
#   29 = ['r', 'ọ̀']
#   2 = ['SP']
#   6 = ['í']
#   22 = ['t', 'u', 'n']
testI = 29
orig_syll = tester[testI]
new_syll = _rm_diacritics_syll(orig_syll, keep='none')
print(orig_syll, new_syll)

print(get_tone(orig_syll))
print(get_tone(new_syll))

minitester = tester[0:50]
ngram_counts = _syll_grams(minitester, dict(), 4)
print(ngram_counts)

untoned = [_rm_diacritics_syll(x) for x in minitester]
print(untoned)
toned = pred_tone(untoned, ngram_counts, 4, print_it=True)
print(toned)

[['a'], ['r', 'a'], ['SP', ' '], ['à'], ['SP', ' '], ['m', 'i'], ['í'], ['SP', ' '], ['d', 'á'], ['SP', ' '], ['ṣ', 'á'], ['k', 'á'], ['SP', ' '], ['b', 'í'], ['i'], ['SP', ' '], ['a'], ['r', 'a'], ['SP', ' '], ['ọ'], ['m', 'ọ'], ['SP', ' '], ['t', 'u', 'n'], ['t', 'u', 'n'], ['SP', ' '], ['n', 'í'], ['SP', ' '], ['à'], ['á'], ['r', 'ọ̀'], ['SP', ' '], ['y', 'ì'], ['í']]
['r', 'ọ̀'] ['r', 'o']
Tones.L
Tones.M
[{'a': {'': {<Tones.M: 0>: 2, <Tones.L: -1>: 2, <Tones.H: 1>: 1}}, 'ra': {'': {<Tones.M: 0>: 2}}, 'SP': {'': {<Tones.M: 0>: 11}}, 'mi': {'': {<Tones.M: 0>: 1}}, 'i': {'': {<Tones.H: 1>: 2, <Tones.M: 0>: 1}}, 'da': {'': {<Tones.H: 1>: 1}}, 'ṣa': {'': {<Tones.H: 1>: 1}}, 'ka': {'': {<Tones.H: 1>: 1}}, 'bi': {'': {<Tones.H: 1>: 1}}, 'ọ': {'': {<Tones.M: 0>: 1}}, 'mọ': {'': {<Tones.M: 0>: 1}}, 'tun': {'': {<Tones.M: 0>: 2}}, 'ni': {'': {<Tones.H: 1>: 1}}, 'rọ': {'': {<Tones.L: -1>: 1}}, 'yi': {'': {<Tones.L: -1>: 1}}}, {'a': {'<.ra': {<Tones.M: 0>: 1}, 'SP.SP': {<T

In [10]:
tester = all_syllables.loc[0:5].copy()
counts = create_syll_grams(tester, keep='none')
print(counts)

detoned = tester.apply(lambda row: rm_diacritics_row(row, keep='none'), axis=1)
tester['Prediction'] = detoned.apply(lambda row: pred_tone(row, counts))

testI = 4
print(tester.loc[testI, 'Syllables'])
print(tester.loc[testI, 'Prediction'])

tester = evaluate(tester, print_it=True)
testI = 4
print(tester.loc[testI, 'Total Words'], tester.loc[testI, 'Sentence'])
wer = (tester['Wrong Words'].sum() / tester['Total Words'].sum()) * 100
print(wer)

[{'a': {'': {<Tones.M: 0>: 5, <Tones.L: -1>: 16, <Tones.H: 1>: 2}}, 'ra': {'': {<Tones.M: 0>: 3}}, 'SP': {'': {<Tones.M: 0>: 66}}, 'mi': {'': {<Tones.M: 0>: 1, <Tones.H: 1>: 2, <Tones.L: -1>: 1}}, 'i': {'': {<Tones.H: 1>: 4, <Tones.M: 0>: 3, <Tones.L: -1>: 9}}, 'da': {'': {<Tones.H: 1>: 1, <Tones.L: -1>: 1}}, 'sa': {'': {<Tones.H: 1>: 1}}, 'ka': {'': {<Tones.H: 1>: 1, <Tones.L: -1>: 1}}, 'bi': {'': {<Tones.H: 1>: 1}}, 'o': {'': {<Tones.M: 0>: 6, <Tones.L: -1>: 4, <Tones.H: 1>: 1}}, 'mo': {'': {<Tones.M: 0>: 1}}, 'tun': {'': {<Tones.M: 0>: 2}}, 'ni': {'': {<Tones.H: 1>: 5, <Tones.M: 0>: 2}}, 'ro': {'': {<Tones.L: -1>: 3}}, 'yi': {'': {<Tones.L: -1>: 2}}, 'gba': {'': {<Tones.H: 1>: 1, <Tones.L: -1>: 1}}, 'ke': {'': {<Tones.M: 0>: 2, <Tones.L: -1>: 2}}, 'ji': {'': {<Tones.L: -1>: 1, <Tones.H: 1>: 2}}, 're': {'': {<Tones.M: 0>: 3, <Tones.L: -1>: 2}}, 'di': {'': {<Tones.M: 0>: 1}}, 'de': {'': {<Tones.M: 0>: 1, <Tones.H: 1>: 3, <Tones.L: -1>: 2}}, 'le': {'': {<Tones.H: 1>: 3, <Tones.L: -1>: 

In [11]:
def dup_rows(df):
    print(f'Original Length: {len(df)}')
    dup_sentences = df.duplicated(subset='Sentence', keep=False)
    dup_indices = dup_sentences[dup_sentences == True]
    print(f'{len(dup_indices)} TOTAL DUPLICATES FOUND')
    for i in dup_indices.index:
        print(f'{i}: {df.loc[i]}')
    dropped = df.drop_duplicates(subset='Sentence',keep='first')
    print(f'New Length: {len(dropped)}')
    return dropped

deduped = dup_rows(all_syllables)

Original Length: 25806
292 TOTAL DUPLICATES FOUND
457: Source                                            IroyinSpeech
ID                                               yo_f_0458.wav
Sentence     Èmi kò ní pẹ́ ṣe ayẹyẹ ọjọ́ ìbí kẹrìndínlọ́gọ́...
Syllables    [[è], [m, i], [SP,  ], [k, ò], [SP,  ], [n, ...
Name: 457, dtype: object
820: Source                                            IroyinSpeech
ID                                               yo_f_0821.wav
Sentence                   Mo dúpẹ́ lọ́wọ́ gómìnà ìpínlẹ̀ Èkó.
Syllables    [[m, o], [SP,  ], [d, ú], [p, ẹ́], [SP,  ], ...
Name: 820, dtype: object
849: Source                                            IroyinSpeech
ID                                               yo_f_0850.wav
Sentence     Ẹ̀wẹ̀, ìròyìn tún fi múlẹ̀ pé ẹnikẹ́ni kò fara...
Syllables    [[ẹ̀], [w, ẹ̀], [SP, ,,  ], [ì], [r, ò], [...
Name: 849, dtype: object
988: Source                                            IroyinSpeech
ID                                  

In [16]:
# split data
curr_df = deduped
# split into 10 folds, make it even across all three datasets
skf = StratifiedKFold(n_splits=10,shuffle=False)
X = curr_df[['ID', 'Sentence', 'Syllables']].to_numpy()
y = curr_df['Source'].to_numpy() # stratify across the datasets

# run predictions for all folds
# NOTE: keep='none' is required for create_syll_grams and rm_diacritics
WERs = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    # if i > 1: break
    n = 0

    print(f"--- Fold {i} ---")
    # get training and testing data
    train_df = curr_df.iloc[train_index].copy()
    test_df = curr_df.iloc[test_index].copy()

    # create n-grams
    start_time = time.time()
    counts = create_syll_grams(train_df, n=n, keep='none')
    end_time = time.time()
    print('N-Gram Timing : ', (end_time-start_time), flush=True)

    # create detoned version (to test predictions)
    start_time = time.time()
    detoned = test_df.apply(lambda row: rm_diacritics_row(row, keep='none'), axis=1)

    # predict tones
    test_df['Prediction'] = detoned.apply(lambda row: pred_tone(row, counts, n=n))
    end_time = time.time()
    print('Prediction Timing : ', (end_time - start_time), flush=True)

    # evaluate
    start_time = time.time()
    test_df = evaluate(test_df, print_it=False)
    wer = (test_df['Wrong Words'].sum() / test_df['Total Words'].sum()) * 100
    end_time = time.time()
    print('Evaluation Timing : ', (end_time - start_time), flush=True)
    print('WER = ', wer, flush=True)
    WERs.append(wer)

    # print(f"  Train: index={train_index}")
    # print(f"  Test:  index={test_index}")

print(float(sum(WERs)) / float(len(WERs)))

--- Fold 0 ---
N-Gram Timing :  2.3906729221343994
Prediction Timing :  0.3559250831604004
Evaluation Timing :  0.26409101486206055
WER =  55.61918894479353
--- Fold 1 ---
N-Gram Timing :  2.320884943008423
Prediction Timing :  0.22409987449645996
Evaluation Timing :  0.2693212032318115
WER =  56.136641088245554
--- Fold 2 ---
N-Gram Timing :  2.4181928634643555
Prediction Timing :  0.1401231288909912
Evaluation Timing :  0.36648988723754883
WER =  55.66643882433356
--- Fold 3 ---
N-Gram Timing :  2.3753111362457275
Prediction Timing :  0.18343496322631836
Evaluation Timing :  0.24946188926696777
WER =  58.1479378516492
--- Fold 4 ---
N-Gram Timing :  2.4237067699432373
Prediction Timing :  0.14975190162658691
Evaluation Timing :  0.37587809562683105
WER =  56.36257632943631
--- Fold 5 ---
N-Gram Timing :  2.3354148864746094
Prediction Timing :  0.2272179126739502
Evaluation Timing :  0.2713601589202881
WER =  51.87834357043788
--- Fold 6 ---
N-Gram Timing :  2.336142063140869
Predicti